In [ ]:
%pip install kaggle --quiet

In [ ]:
import kagglehub
kagglehub.login()
path = kagglehub.competition_download('playground-series-s6e4')

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [ ]:

train = pd.read_csv(path+'/train.csv')
test_data = pd.read_csv(path+'/test.csv')


In [ ]:

cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
            'Season', 'Irrigation_Type', 'Water_Source',
            'Mulching_Used', 'Region']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    encoders[col] = le

for col in cat_cols:
    test_data[col] = encoders[col].transform(test_data[col])


In [ ]:

mapping = {'Low':0, 'Medium':1, 'High':2}
train['Irrigation_Need'] = train['Irrigation_Need'].map(mapping)


In [ ]:

num_cols = train.select_dtypes(include=['float64']).columns

for col in num_cols:
    train[col] = train[col].fillna(train[col].mean())

scaler = StandardScaler()
train[num_cols] = scaler.fit_transform(train[num_cols])

for col in num_cols:
    test_data[col] = test_data[col].fillna(test_data[col].mean())

test_data[num_cols] = scaler.transform(test_data[num_cols])


In [ ]:

class IrrigationDataset(Dataset):
    def __init__(self, data):
        self.X = data.drop(['id','Irrigation_Need'], axis=1).values
        self.y = data['Irrigation_Need'].values

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.long)
        )


In [ ]:

train_data, val_data = train_test_split(train, test_size=0.2, random_state=42)

train_dataset = IrrigationDataset(train_data)
val_dataset = IrrigationDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


In [ ]:

class IrrigationModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,3)
        )

    def forward(self,x):
        return self.net(x)


In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = IrrigationModel(train_dataset.X.shape[1]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)


In [ ]:

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0,0,0

    for X,y in train_loader:
        X,y = X.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = out.argmax(1)
        correct += (pred==y).sum().item()
        total += y.size(0)

    print(f"Epoch {epoch+1} Loss:{total_loss:.4f} Acc:{correct/total:.4f}")


In [ ]:

class TestDataset(Dataset):
    def __init__(self,data):
        self.X = data.drop(['id'],axis=1).values

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return torch.tensor(self.X[idx],dtype=torch.float32)

test_dataset = TestDataset(test_data)
test_loader = DataLoader(test_dataset,batch_size=32)


In [ ]:

model.eval()
preds = []

with torch.no_grad():
    for X in test_loader:
        X = X.to(device)
        out = model(X)
        pred = out.argmax(1)
        preds.extend(pred.cpu().numpy())


In [ ]:

idx_to_label = {0:'Low',1:'Medium',2:'High'}
pred_labels = [idx_to_label[i] for i in preds]

submission = pd.DataFrame({
    'id': test_data['id'],
    'Irrigation_Need': pred_labels
})

submission.to_csv('submission.csv',index=False)
submission.head()
